# Modern Portfolio Theory from Scratch: The Efficient Frontier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/quant-finance/modern_portfolio_theory.ipynb)

Build the Markowitz efficient frontier using matrix algebra and Monte Carlo simulation.

**Blog post:** [Modern Portfolio Theory from Scratch](https://sesen.ai/blog/modern-portfolio-theory-markowitz-efficient-frontier)

**Key paper:** Markowitz, H. (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77-91.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Setup: Assets and Covariance Structure

We define a 4-asset universe with expected annual returns and a covariance matrix.
Volatilities range from 15% to 20%, with correlations chosen to produce a well-spread frontier.

In [ ]:
asset_names = ["US Equity", "Intl Equity", "Real Estate", "Commodities"]
mu = np.array([0.11, 0.16, 0.06, 0.04])  # expected annual returns

# Covariance matrix: vol = 15%, 20%, 17%, 16%
# Correlations: US-Intl 0.6, US-RE 0.3, US-Com 0.0, Intl-RE 0.35, Intl-Com 0.1, RE-Com 0.15
cov_matrix = np.array([
    [0.0225, 0.0180, 0.0077, 0.0000],   # US Equity (15%)
    [0.0180, 0.0400, 0.0119, 0.0032],   # Intl Equity (20%)
    [0.0077, 0.0119, 0.0289, 0.0043],   # Real Estate (17%)
    [0.0000, 0.0032, 0.0043, 0.0256],   # Commodities (16%)
])

n_assets = len(mu)
print("Individual asset Sharpe ratios:")
for i, name in enumerate(asset_names):
    vol = np.sqrt(cov_matrix[i, i])
    print(f"  {name}: {mu[i]/vol:.2f} (ret={mu[i]:.1%}, vol={vol:.1%})")

## 2. Quick Win: The R Code Translation

The original R code computes portfolio return and variance using matrix algebra for a 3-asset portfolio.

In [ ]:
# Direct translation of the original R code
w = np.array([1/3, 1/3, 1/3])
mu_3 = np.array([0.01, 0.04, 0.02])
cov_3 = np.array([
    [0.10,  0.30,  0.10],
    [0.30,  0.15, -0.20],
    [0.10, -0.20,  0.08],
])

port_return = w @ mu_3           # E[Rp] = w' * mu
port_variance = w @ cov_3 @ w   # Var(Rp) = w' * Sigma * w
print(f"Portfolio return:   {port_return:.4f}")
print(f"Portfolio variance: {port_variance:.4f}")
print(f"Portfolio std dev:  {np.sqrt(port_variance):.4f}")

## 3. Monte Carlo Simulation of the Efficient Frontier

In [ ]:
n_portfolios = 50_000

# Generate random portfolio weights using the Dirichlet distribution
# Each draw is non-negative and sums to 1 (long-only constraint)
all_weights = np.random.dirichlet(np.ones(n_assets), size=n_portfolios)

# Compute return and volatility for each portfolio
port_returns = all_weights @ mu
port_volatilities = np.sqrt(
    np.array([w @ cov_matrix @ w for w in all_weights])
)
sharpe_ratios = port_returns / port_volatilities

# Find special portfolios from the MC simulation
min_vol_idx = np.argmin(port_volatilities)
max_sharpe_idx = np.argmax(sharpe_ratios)

print(f"Min-variance portfolio:  return={port_returns[min_vol_idx]:.2%}, "
      f"vol={port_volatilities[min_vol_idx]:.2%}")
print(f"Max-Sharpe portfolio:    return={port_returns[max_sharpe_idx]:.2%}, "
      f"vol={port_volatilities[max_sharpe_idx]:.2%}, "
      f"Sharpe={sharpe_ratios[max_sharpe_idx]:.2f}")

In [ ]:
# Analytical solutions (unconstrained)
ones = np.ones(n_assets)
inv_cov = np.linalg.inv(cov_matrix)

# Minimum-variance portfolio
w_mv = inv_cov @ ones / (ones @ inv_cov @ ones)
mv_ret = w_mv @ mu
mv_vol = np.sqrt(w_mv @ cov_matrix @ w_mv)

# Tangency (max Sharpe) portfolio
w_tan = inv_cov @ mu / (ones @ inv_cov @ mu)
tan_ret = w_tan @ mu
tan_vol = np.sqrt(w_tan @ cov_matrix @ w_tan)
tan_sharpe = tan_ret / tan_vol

print("Minimum-variance weights (analytical):")
for name, weight in zip(asset_names, w_mv):
    print(f"  {name}: {weight:.1%}")
print(f"Return: {mv_ret:.2%}, Vol: {mv_vol:.2%}")

print(f"\nMax-Sharpe (tangency) weights:")
for name, weight in zip(asset_names, w_tan):
    print(f"  {name}: {weight:.1%}")
print(f"Return: {tan_ret:.2%}, Vol: {tan_vol:.2%}, Sharpe: {tan_sharpe:.2f}")

## 4. Plotting the Efficient Frontier

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(port_volatilities * 100, port_returns * 100,
                     c=sharpe_ratios, cmap='viridis', alpha=0.3, s=5, edgecolors='none')
ax.scatter(mv_vol * 100, mv_ret * 100, c='red', marker='*', s=300, zorder=5,
           edgecolors='black', linewidths=0.5, label=f'Min Variance ({mv_vol:.1%}, {mv_ret:.1%})')
ax.scatter(tan_vol * 100, tan_ret * 100, c='gold', marker='*', s=300, zorder=5,
           edgecolors='black', linewidths=0.5, label=f'Max Sharpe ({tan_vol:.1%}, {tan_ret:.1%})')

# Plot individual assets
asset_vols = np.sqrt(np.diag(cov_matrix)) * 100
asset_rets = mu * 100
for i, name in enumerate(asset_names):
    ax.scatter(asset_vols[i], asset_rets[i], marker='D', s=80, c='white',
               edgecolors='black', linewidths=1.5, zorder=6)
    ax.annotate(name, (asset_vols[i], asset_rets[i]), textcoords='offset points',
                xytext=(8, 5), fontsize=9)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Sharpe Ratio', fontsize=11)
ax.set_xlabel('Annualised Volatility (%)', fontsize=12)
ax.set_ylabel('Annualised Return (%)', fontsize=12)
ax.set_title('The Efficient Frontier: 50,000 Random Portfolios', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Capital Market Line

In [ ]:
# Extract efficient frontier
n_bins = 100
vol_bins = np.linspace(port_volatilities.min(), port_volatilities.max(), n_bins)
frontier_returns = []
frontier_vols = []

for i in range(len(vol_bins) - 1):
    mask = (port_volatilities >= vol_bins[i]) & (port_volatilities < vol_bins[i + 1])
    if mask.sum() > 0:
        best_idx = np.argmax(port_returns[mask])
        frontier_returns.append(port_returns[mask][best_idx])
        frontier_vols.append(port_volatilities[mask][best_idx])

# Plot with CML
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(port_volatilities * 100, port_returns * 100, c='lightgray', alpha=0.2, s=3, edgecolors='none')
ax.plot(np.array(frontier_vols) * 100, np.array(frontier_returns) * 100,
        'b-', linewidth=2, label='Efficient Frontier')

# CML from risk-free rate (0) through tangency portfolio
cml_vols = np.linspace(0, 0.25, 100)
cml_returns = (tan_ret / tan_vol) * cml_vols
ax.plot(cml_vols * 100, cml_returns * 100, 'r--', linewidth=2, label='Capital Market Line')

ax.scatter(0, 0, c='green', marker='s', s=100, zorder=6, label='Risk-Free Rate (0%)')
ax.scatter(mv_vol * 100, mv_ret * 100, c='red', marker='*', s=300, zorder=6,
           edgecolors='black', linewidths=0.5, label='Min Variance')
ax.scatter(tan_vol * 100, tan_ret * 100, c='gold', marker='*', s=300, zorder=6,
           edgecolors='black', linewidths=0.5, label='Tangency Portfolio')

ax.set_xlabel('Annualised Volatility (%)', fontsize=12)
ax.set_ylabel('Annualised Return (%)', fontsize=12)
ax.set_title('Capital Market Line', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(-0.5, 25)
ax.set_ylim(-0.5, 22)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Portfolio Weight Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(n_assets)
width = 0.6
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

axes[0].bar(x, w_mv * 100, width, color=colors)
axes[0].set_title('Minimum Variance Portfolio', fontsize=13)
axes[0].set_ylabel('Weight (%)', fontsize=11)
axes[0].set_xticks(x)
axes[0].set_xticklabels(asset_names, rotation=15)
for i, v in enumerate(w_mv * 100):
    axes[0].text(i, v + 1 if v >= 0 else v - 3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

axes[1].bar(x, w_tan * 100, width, color=colors)
axes[1].set_title('Maximum Sharpe Portfolio', fontsize=13)
axes[1].set_ylabel('Weight (%)', fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels(asset_names, rotation=15)
for i, v in enumerate(w_tan * 100):
    axes[1].text(i, v + 1 if v >= 0 else v - 3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

for ax in axes:
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Optimal Portfolio Allocations (Unconstrained)', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Exercises

1. **Change the covariance structure.** Set all off-diagonal covariances to zero and re-run the Monte Carlo. How does the frontier shape change?

2. **Add a constraint.** Require each asset to have at least 5% weight. How does this affect the minimum-variance portfolio?

3. **Use real data.** Download returns for SPY, EFA, VNQ, and DJP from Yahoo Finance using `yfinance`, compute the sample mean and covariance, and plot the frontier with real numbers.

4. **Add a risk-free rate.** Set the risk-free rate to 2% and recompute the tangency portfolio and CML. How do the weights change?